## CS-345 Term Project 
Dacovney Brochu and Nuri Shawesh

### Project Overview
(Description of the project and its goal)

### Data Set Description
(Basic information on the data set being used + probably a snippet on how the data isn't real)

Setting up the dataset:

In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

salesDataset = pd.read_csv('sales.csv')
productsDataset = pd.read_csv('products.csv')
calendarDataset = pd.read_csv('calendar.csv')
storesDataset = pd.read_csv('stores.csv')
customersDataset = pd.read_csv('customers.csv')

print("salesDataset:", salesDataset.shape)
print("productsDataset:", productsDataset.shape)
print("calendarDataset:", calendarDataset.shape)
print("storesDataset:", storesDataset.shape)
print("customersDataset:", customersDataset.shape)

salesDataset: (1000000, 11)
productsDataset: (200, 6)
calendarDataset: (731, 6)
storesDataset: (100, 5)
customersDataset: (50000, 5)


Creating a masterset:

In [27]:
masterDataset = (salesDataset.merge(productsDataset, on='product_id', how='left').merge(storesDataset, on='store_id', how='left').merge(customersDataset, on='customer_id', how='left'))
print(masterDataset.shape)
print(masterDataset.columns.tolist())
masterDataset.head(10)

(1000000, 24)
['order_id', 'order_date', 'product_id', 'store_id', 'customer_id', 'quantity', 'unit_price', 'discount', 'revenue', 'cost', 'profit', 'product_name', 'brand', 'category', 'cocoa_percent', 'weight_g', 'store_name', 'city', 'country', 'store_type', 'age', 'gender', 'loyalty_member', 'join_date']


,order_id,order_date,product_id,store_id,customer_id,quantity,unit_price,discount,revenue,cost,...,cocoa_percent,weight_g,store_name,city,country,store_type,age,gender,loyalty_member,join_date
0,0RD00000001,2023-01-07,P0080,S093,C040749,5,14.43,0.15,61.33,42.77,...,70.0,200.0,Chocolate Store 93,Sydney,UK,Airport,44,Male,1,2021-11-17
1,0RD00000002,2023-10-22,P0173,S065,C020161,3,12.01,0.00,36.03,19.06,...,60.0,50.0,Chocolate Store 65,New York,Australia,Retail,63,Female,1,2023-07-03
2,0RD00000003,2023-05-07,P0115,S078,C048069,2,10.02,0.00,20.04,10.29,...,90.0,50.0,Chocolate Store 78,London,UK,Airport,35,Male,1,2023-10-09
3,0RD00000004,2024-06-23,P0186,S088,C047901,2,14.66,0.10,26.39,16.35,...,60.0,50.0,Chocolate Store 88,Toronto,USA,Retail,37,Female,1,2023-05-30
4,0RD00000005,2024-09-24,P0197,S054,C033950,1,12.34,0.00,12.34,7.94,...,90.0,120.0,Chocolate Store 54,London,Canada,Online,57,Female,0,2021-08-20
5,0RD00000006,2024-03-29,P0160,S089,C008918,4,13.52,0.00,54.08,36.59,...,90.0,100.0,Chocolate Store 89,Paris,Canada,Online,35,Male,0,2024-11-24
6,0RD00000007,2023-02-26,P0062,S024,C002897,1,11.97,0.10,10.77,7.16,...,80.0,120.0,Chocolate Store 24,Paris,France,Online,55,Female,0,2025-09-27
7,0RD00000008,2023-11-03,P0111,S085,C038072,5,4.62,0.00,23.10,16.15,...,80.0,50.0,Chocolate Store 85,Melbourne,USA,Online,36,Female,0,2023-03-29
8,0RD00000009,2024-10-11,P0135,S029,C003786,4,7.88,0.00,31.52,19.90,...,80.0,200.0,Chocolate Store 29,Melbourne,Australia,Online,63,Female,1,2024-01-11
9,0RD00000010,2023-12-17,P0069,S056,C043148,3,8.88,0.00,26.64,18.19,...,90.0,120.0,Chocolate Store 56,Melbourne,Germany,Online,26,Female,0,2022-01-27


### Cleaning the Data

* Edit this to be more purposeful and professional sounding on why we cut these things

#Consider cutting out data that falls into these 3 categories:
1. The target, so what we're trying to figure out. In this case we are trying to figure out if a product will be high profit or not. Anything that would reveal or leak this information to the model should be considered to be cut out
2. Useless identifiers like the store ID, ProdID, CustomerID etc. this doesn't really help in knowing if a product is high profit
3. Features that might just add noise. City should be dropped because it's inaccurate. Country has a argument to keep but could be added noise. JoinDate is irrelevant

In [28]:
columnsToDrop = [
    "order_id",
    "store_id",
    "product_id",
    "customer_id",
    "city",
    "join_date"
]
masterDataset = masterDataset.drop(columns=columnsToDrop)
print(masterDataset.shape)
print(masterDataset.columns.tolist())
masterDataset.head(10)

(1000000, 18)
['order_date', 'quantity', 'unit_price', 'discount', 'revenue', 'cost', 'profit', 'product_name', 'brand', 'category', 'cocoa_percent', 'weight_g', 'store_name', 'country', 'store_type', 'age', 'gender', 'loyalty_member']


,order_date,quantity,unit_price,discount,revenue,cost,profit,product_name,brand,category,cocoa_percent,weight_g,store_name,country,store_type,age,gender,loyalty_member
0,2023-01-07,5,14.43,0.15,61.33,42.77,18.56,Praline Chocolate 70%,Hershey,White,70.0,200.0,Chocolate Store 93,UK,Airport,44,Male,1
1,2023-10-22,3,12.01,0.00,36.03,19.06,16.97,Dark Chocolate 60%,Lindt,Praline,60.0,50.0,Chocolate Store 65,Australia,Retail,63,Female,1
2,2023-05-07,2,10.02,0.00,20.04,10.29,9.75,Milk Chocolate 90%,Hershey,Milk,90.0,50.0,Chocolate Store 78,UK,Airport,35,Male,1
3,2024-06-23,2,14.66,0.10,26.39,16.35,10.04,Dark Chocolate 60%,Godiva,Praline,60.0,50.0,Chocolate Store 88,USA,Retail,37,Female,1
4,2024-09-24,1,12.34,0.00,12.34,7.94,4.40,Truffle Chocolate 90%,Hershey,Truffle,90.0,120.0,Chocolate Store 54,Canada,Online,57,Female,0
5,2024-03-29,4,13.52,0.00,54.08,36.59,17.49,Praline Chocolate 90%,Hershey,Dark,90.0,100.0,Chocolate Store 89,Canada,Online,35,Male,0
6,2023-02-26,1,11.97,0.10,10.77,7.16,3.61,Truffle Chocolate 80%,Mars,Dark,80.0,120.0,Chocolate Store 24,France,Online,55,Female,0
7,2023-11-03,5,4.62,0.00,23.10,16.15,6.95,White Chocolate 80%,Cadbury,Dark,80.0,50.0,Chocolate Store 85,USA,Online,36,Female,0
8,2024-10-11,4,7.88,0.00,31.52,19.90,11.62,White Chocolate 80%,Godiva,Praline,80.0,200.0,Chocolate Store 29,Australia,Online,63,Female,1
9,2023-12-17,3,8.88,0.00,26.64,18.19,8.45,White Chocolate 90%,Cadbury,Milk,90.0,120.0,Chocolate Store 56,Germany,Online,26,Female,0


In [29]:
threshold = masterDataset['profit'].quantile(0.75)
print('threshold:', threshold)

masterDataset['high_profit'] = (masterDataset['profit'] > threshold).astype(int)
print(masterDataset['high_profit'].value_counts())
print(masterDataset['high_profit'].value_counts(normalize=True))
print(masterDataset.columns.tolist())

threshold: 14.17
high_profit
0    750077
1    249923
Name: count, dtype: int64
high_profit
0    0.750077
1    0.249923
Name: proportion, dtype: float64
['order_date', 'quantity', 'unit_price', 'discount', 'revenue', 'cost', 'profit', 'product_name', 'brand', 'category', 'cocoa_percent', 'weight_g', 'store_name', 'country', 'store_type', 'age', 'gender', 'loyalty_member', 'high_profit']


In [30]:
dataLeakColumnsToDrop = [
    "profit",
    "cost",
    "revenue"
]
masterDataset = masterDataset.drop(columns=dataLeakColumnsToDrop)
print(masterDataset.shape)
print(masterDataset.columns.tolist())
masterDataset.head(10)

(1000000, 16)
['order_date', 'quantity', 'unit_price', 'discount', 'product_name', 'brand', 'category', 'cocoa_percent', 'weight_g', 'store_name', 'country', 'store_type', 'age', 'gender', 'loyalty_member', 'high_profit']


,order_date,quantity,unit_price,discount,product_name,brand,category,cocoa_percent,weight_g,store_name,country,store_type,age,gender,loyalty_member,high_profit
0,2023-01-07,5,14.43,0.15,Praline Chocolate 70%,Hershey,White,70.0,200.0,Chocolate Store 93,UK,Airport,44,Male,1,1
1,2023-10-22,3,12.01,0.00,Dark Chocolate 60%,Lindt,Praline,60.0,50.0,Chocolate Store 65,Australia,Retail,63,Female,1,1
2,2023-05-07,2,10.02,0.00,Milk Chocolate 90%,Hershey,Milk,90.0,50.0,Chocolate Store 78,UK,Airport,35,Male,1,0
3,2024-06-23,2,14.66,0.10,Dark Chocolate 60%,Godiva,Praline,60.0,50.0,Chocolate Store 88,USA,Retail,37,Female,1,0
4,2024-09-24,1,12.34,0.00,Truffle Chocolate 90%,Hershey,Truffle,90.0,120.0,Chocolate Store 54,Canada,Online,57,Female,0,0
5,2024-03-29,4,13.52,0.00,Praline Chocolate 90%,Hershey,Dark,90.0,100.0,Chocolate Store 89,Canada,Online,35,Male,0,1
6,2023-02-26,1,11.97,0.10,Truffle Chocolate 80%,Mars,Dark,80.0,120.0,Chocolate Store 24,France,Online,55,Female,0,0
7,2023-11-03,5,4.62,0.00,White Chocolate 80%,Cadbury,Dark,80.0,50.0,Chocolate Store 85,USA,Online,36,Female,0,0
8,2024-10-11,4,7.88,0.00,White Chocolate 80%,Godiva,Praline,80.0,200.0,Chocolate Store 29,Australia,Online,63,Female,1,0
9,2023-12-17,3,8.88,0.00,White Chocolate 90%,Cadbury,Milk,90.0,120.0,Chocolate Store 56,Germany,Online,26,Female,0,0


In [32]:
# Remove order_date to no longer be a string
masterDataset['order_date'] = pd.to_datetime(masterDataset['order_date'])

masterDataset['order_month'] = masterDataset['order_date'].dt.month
masterDataset['order_dayofweek'] = masterDataset['order_date'].dt.dayofweek

masterDataset = masterDataset.drop(columns=['order_date'])

In [33]:
# Encode Categorical Columns
masterDataset = pd.get_dummies(
    masterDataset,
    drop_first=True
)

In [34]:
# Separate features and target
X = masterDataset.drop(columns=['high_profit'])
y = masterDataset['high_profit']

### Creating Train/Test Split

In [35]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Train Random Forest Baseline

In [37]:
from sklearn.ensemble import RandomForestClassifier

rfModel = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rfModel.fit(X_train, y_train)

# Prediction
rfPredictions = rfModel.predict(X_test)

In [38]:
#Evaluation
from sklearn.metrics import classification_report, accuracy_score

print ("Random Forest Accuracy:")
print(accuracy_score(y_test, rfPredictions))

print ("\nRandom Forest Classification Report:")
print(classification_report(y_test, rfPredictions))

Random Forest Accuracy:
0.937635

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96    150015
           1       0.88      0.87      0.87     49985

    accuracy                           0.94    200000
   macro avg       0.92      0.91      0.92    200000
weighted avg       0.94      0.94      0.94    200000



### Train Gradient Boosting Model

In [41]:
# Drop rows that are missing data, as GradientBoosting does not handle missing values
X_train = X_train.dropna()
y_train = y_train.loc[X_train.index]

X_test = X_test.dropna()
y_test = y_test.loc[X_test.index]

In [42]:
from sklearn.ensemble import GradientBoostingClassifier

gbModel = GradientBoostingClassifier(
    random_state=42
)

gbModel.fit(X_train, y_train)

# Prediction
gbPredictions = gbModel.predict(X_test)

In [43]:
#Evaluation

print ("Gradient Boosting Accuracy:")
print(accuracy_score(y_test, gbPredictions))

print ("\nGradient Boosting Classification Report:")
print(classification_report(y_test, gbPredictions))

Gradient Boosting Accuracy:
0.9395966715812009

Gradient Boosting Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96    148555
           1       0.87      0.89      0.88     49497

    accuracy                           0.94    198052
   macro avg       0.92      0.92      0.92    198052
weighted avg       0.94      0.94      0.94    198052



## Discussion